## Imports

In [13]:
from pathlib import Path
import json
import pickle
import math
from collections import Counter, defaultdict
from tokenizers import Tokenizer
import re

---

## Loading all the files from preprocessing

In [2]:
PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR/"data"/"processed"

hf_tokenizer = Tokenizer.from_file(str(PROCESSED_DIR/"tokenizer.json"))

with open(PROCESSED_DIR/"encoded_lines.pkl", "rb") as f:
    encoded_lines = pickle.load(f)
    
with open(PROCESSED_DIR/"vocab.json", "r", encoding="utf-8") as f:
    vocab_data = json.load(f)
    
word_to_id = vocab_data["word_to_id"]
id_to_word = {idx: word for word, idx in word_to_id.items()}

vocab_size = len(word_to_id)

bos_id = word_to_id["<bos>"]
eos_id = word_to_id["<eos>"]
unk_id = word_to_id["<unk>"]
pad_id = word_to_id["<pad>"]

print("Vocabulary size:", vocab_size)
print()
for split_name, sequences in encoded_lines.items():
    print(split_name, "sequences:", len(sequences))
print()
print("Example encoded sequence:")
print(encoded_lines["train"][0][:20])

Vocabulary size: 46165

test sequences: 2185
train sequences: 17556
validation sequences: 1841

Example encoded sequence:
[2, 21646, 119, 3840, 152, 44, 1, 4345, 23, 749, 44, 33778, 5, 7440, 6, 3840, 7, 4, 23730, 152]


---

## Ngram Model

so this is a trigram model but with:  
- interpolation: so combining trigram, bigram and unigram probabilities
- additive/laplace smoothing to avoid zero probabilities
- backoff prediction, falling back to shorter contexts when needed

counts dict:
```
{
    context : {
        next_token : frequency
    }
}

{
    ("I", "love"): {
        "NLP": 4,
        "pizza": 2
    }
}
```  

and counts_totals:  
```
{
    context : total_predictions
}

{
    ("I","love"): 6
}
```

In [3]:
class NGramLanguageModel:
    def __init__(self, order=3, alpha=0.0001): # so we use two previous tokens to predict the next token
        self.order = order
        self.alpha = alpha # additive smoothing to prevent zero probabilities
        self.counts = defaultdict(Counter) # counts[context][next_token] = freq
        self.context_totals = Counter() # total num of predictions observed for each context
        self.vocab_size = None
        self._top_cache = {} # to avoid repeatedly computing
        self.lambdas = {2: 0.6, 1: 0.3, 0: 0.1} # weights used for interpolation
        
    def fit(self, sequences, vocab_size):
        # first reseting structures to remove old model statistics
        self.counts.clear()
        self.context_totals.clear()
        self._top_cache.clear()
        
        self.vocab_size = vocab_size
        
        for sequence in sequences: # every sentence
            for i in range(1, len(sequence)): # cuz the first token is bos, which is a context only
                target = sequence[i]
                
                # for context
                start_in = max(0, i - (self.order - 1)) # like for i=3 is  3 - (3-1) = 1
                context = tuple(sequence[start_in:i]) # [1:3] so 1 and 2
                
                # and then we count every suffix of the context so that the model falls back to shorter context
                for length in range(len(context)+1):
                    if length == 0:
                        suffix=()
                    else:
                        suffix = context[-length:]
                        
                    self.counts[suffix][target] += 1
                    self.context_totals[suffix] += 1

        return self
     
    def _top(self, suffix, k): # to return the most common next tokens for a context
        key = (suffix, k)
        if key not in self._top_cache: # if it is not already there
            counter = self.counts.get(suffix)

            self._top_cache[key] = ([t for t, _ in counter.most_common(k)] if counter else []) # puth the k most common ones in there

        return self._top_cache[key]
    
    def probability(self, context, target): # P(target | context)

        context = tuple(context[-(self.order - 1):]) # so last two tokens 

        p = 0.0 # probability
        w_sum = 0.0 # sum of weights 

        for length in range(len(context), -1, -1): # startnig from the recent ones
            suffix = context[-length:] if length else ()
            total = self.context_totals.get(suffix, 0)
            if total == 0:
                continue

            w = self.lambdas[length] # we get the interpolation weight
            count = self.counts[suffix].get(target, 0) # we get the frequency

            p += w * ((count + self.alpha) / (total + self.alpha * self.vocab_size)) # so smoothed probability multiplied by weight
            #  like 0.6 × trigram + 0.3 × bigram + 0.1 × unigram
            w_sum += w

        return p / w_sum if w_sum else 1.0 / self.vocab_size # so normalizes when some contexts were skipped
        
    def predict_next(self, context, top_k=5):
        context = tuple(context[-(self.order - 1):])

        results = []
        for length in range(len(context), -1, -1):
            suffix = context[-length:] if length else ()

            for token_id in self._top(suffix, top_k):
                if token_id not in results:
                    results.append(token_id)
                    if len(results) == top_k: # to return top k
                        return results

        return results

---

## Training the model

In [4]:
# creating the model
ngram_model = NGramLanguageModel(order=3, alpha=0.0001)

# learning the n-gram counts from training
ngram_model.fit(sequences=encoded_lines["train"], vocab_size=vocab_size)

### Making predictions

In [6]:
example_sequence = encoded_lines["train"][9]

# using all the tokens except the final one as a prefix
prefix_ids = example_sequence[:-1]

# five most frequent continuations
predicted_ids = ngram_model.predict_next(prefix_ids, top_k=5)

print("Input sequences:")
print(" ".join(id_to_word[token_id] for token_id in prefix_ids[-10:]))

print("\nPredicted next words:")
for token_id in predicted_ids:
    print(id_to_word[token_id])

Input sequences:
darker and more somber than that of its predecessor .

Predicted next words:
<eos>
Leaf
Stuart
"
The


---

## Evaluation

In [ ]:

def evaluate_model(model, sequences, top_k=5):
    # initializing counters
    unk_count = 0
    known_count = 0 # normal vocabulary tokens
    total_log_probability = 0.0 # for perplexity
    total_predictions = 0
    correct_top1 = 0 # how often predictions[0] == target
    correct_topk = 0 # how often target is in predictions

    for sequence in sequences:
        for i in range(1, len(sequence)):

            target = sequence[i]
            start = max(0, i - (model.order - 1))
            context = sequence[start:i]

            # for counting unknown and known targets
            if target == unk_id:
                unk_count += 1
            else:
                known_count += 1

            probability = model.probability(context, target) # p for all targets
            total_log_probability += math.log(probability)
            total_predictions += 1

            # accuracy for known ones
            if target != unk_id:
                predictions = model.predict_next(context, top_k=top_k)

                if predictions and target == predictions[0]:
                    correct_top1 += 1

                if target in predictions:
                    correct_topk += 1

    # for only known targets
    top1_accuracy = (correct_top1 / known_count if known_count else 0.0) 
    topk_accuracy = (correct_topk / known_count if known_count else 0.0)

    # and perplexity
    if total_predictions > 0:
        average_negative_log_probability = (-total_log_probability / total_predictions)
        perplexity = math.exp(average_negative_log_probability)
        unk_rate = unk_count / total_predictions
    else:
        perplexity = float("nan")
        unk_rate = 0.0

    return { # metrics : values
        "perplexity": perplexity,
        "top1_accuracy": top1_accuracy,
        f"top{top_k}_accuracy": topk_accuracy,
        "unk_rate": unk_rate,
        "number_of_predictions": total_predictions
    }

### on validation set

In [8]:
validation_results = evaluate_model(ngram_model, encoded_lines["validation"])

print("Validation results")
print("~~~~~~~~~~~~~~~~~~~")

for metric, value in validation_results.items():
    print(f"{metric}: {value}")

Validation results
~~~~~~~~~~~~~~~~~~~
perplexity: 383.40241170781104
top1_accuracy: 0.1788090332236471
top5_accuracy: 0.3582382548615596
unk_rate: 0.053766465555706446
number_of_predictions: 206188


### to find the best alpha

In [11]:

alpha_values = [0.000001, 0.00001, 0.00005, 0.0001, 0.0005, 0.001, 0.01]

lambda_options = [ {2: 0.6, 1: 0.3, 0: 0.1}, {2: 0.5, 1: 0.3, 0: 0.2}, {2: 0.7, 1: 0.2, 0: 0.1}, {2: 0.4, 1: 0.4, 0: 0.2}]

# best of everything we wanna find 
best_perplexity = float("inf")
best_alpha = None
best_lambdas = None

validation_scores = []

for alpha in alpha_values:
    for lambdas in lambda_options:
        ngram_model.alpha = alpha
        ngram_model.lambdas = lambdas.copy()

        results = evaluate_model(ngram_model, encoded_lines["validation"])

        ppl = results["perplexity"]

        validation_scores.append({"alpha": alpha, "lambdas": lambdas.copy(), "perplexity": ppl})

        print(f"alpha ={alpha}, "
              f"lambdas ={lambdas}, "
              f"perplexity ={ppl:.4f}")

        if ppl < best_perplexity:
            best_perplexity = ppl
            best_alpha = alpha
            best_lambdas = lambdas.copy()

# then update
ngram_model.alpha = best_alpha
ngram_model.lambdas = best_lambdas.copy()

print("\n~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
print("\nBest alpha:", best_alpha)
print("Best lambdas:", best_lambdas)
print("Best validation perplexity:", best_perplexity)


alpha =1e-06, lambdas ={2: 0.6, 1: 0.3, 0: 0.1}, perplexity =404.9834
alpha =1e-06, lambdas ={2: 0.5, 1: 0.3, 0: 0.2}, perplexity =365.7106
alpha =1e-06, lambdas ={2: 0.7, 1: 0.2, 0: 0.1}, perplexity =426.7443
alpha =1e-06, lambdas ={2: 0.4, 1: 0.4, 0: 0.2}, perplexity =356.2698
alpha =1e-05, lambdas ={2: 0.6, 1: 0.3, 0: 0.1}, perplexity =390.9832
alpha =1e-05, lambdas ={2: 0.5, 1: 0.3, 0: 0.2}, perplexity =359.0918
alpha =1e-05, lambdas ={2: 0.7, 1: 0.2, 0: 0.1}, perplexity =411.1820
alpha =1e-05, lambdas ={2: 0.4, 1: 0.4, 0: 0.2}, perplexity =350.4365
alpha =5e-05, lambdas ={2: 0.6, 1: 0.3, 0: 0.1}, perplexity =382.3110
alpha =5e-05, lambdas ={2: 0.5, 1: 0.3, 0: 0.2}, perplexity =357.0725
alpha =5e-05, lambdas ={2: 0.7, 1: 0.2, 0: 0.1}, perplexity =402.0401
alpha =5e-05, lambdas ={2: 0.4, 1: 0.4, 0: 0.2}, perplexity =348.3720
alpha =0.0001, lambdas ={2: 0.6, 1: 0.3, 0: 0.1}, perplexity =383.4024
alpha =0.0001, lambdas ={2: 0.5, 1: 0.3, 0: 0.2}, perplexity =360.5220
alpha =0.0001, lam

## Evaluating on test

In [ ]:
test_results = evaluate_model(ngram_model, encoded_lines["test"])

print("Final n-gram test results")
print("~~~~~~~~~~~~~~~~~~~~~~~~~")

for metric, value in test_results.items():
    if "accuracy" in metric:
        print(f"{metric}: {value:.4%}") # percentage format
    elif isinstance(value, float): # if it is float
        print(f"{metric}: {value:.4f}")
    else:
        print(f"{metric}: {value}")

Final n-gram test results
~~~~~~~~~~~~~~~~~~~~~~~~~
perplexity: 338.6003
top1_accuracy: 17.8592%
top5_accuracy: 35.7903%
unk_rate: 0.0596
number_of_predictions: 232065


----

## Trying out with the real data

In [14]:

def normalize_for_model(text):

    text = re.sub(r'(?<=\w)([.,;:!?)"])(?=\s|$)', r' \1', text) # to add a space before the punctuation
    text = re.sub(r'(^|\s)([("])(?=\w)', r'\1\2 ', text) # to add a space after opening punctuation
    text = re.sub(r"(\w)'s\b", r"\1 's", text) # to separate possessives
    
    return re.sub(r"\s+", " ", text).strip() # to first replace multiple spaces with one and then removing the spaces at the beginning and end


def autocomplete(text, model, top_k=5):

    normalized_text = normalize_for_model(text)

    prefix_ids = [bos_id] + hf_tokenizer.encode(normalized_text).ids

    special_ids = {bos_id, eos_id, pad_id, unk_id}

    predictions = model.predict_next(prefix_ids, top_k=top_k + 20)

    suggestions = []
    for token_id in predictions:
        if token_id not in special_ids:
            suggestions.append(id_to_word[token_id])

        if len(suggestions) == top_k:
            break

    return suggestions

In [15]:
test_prefixes = ["", "The game was", "He went to the", "The city of", "She said.", "The player's"]

for prefix in test_prefixes:
    print("Prefix:", repr(prefix))
    print("Suggestions:", autocomplete(prefix, ngram_model))
    print()

Prefix: ''
Suggestions: ['The', 'In', '"', 'On', 'A']

Prefix: 'The game was'
Suggestions: ['released', '"', 'the', 'played', 'developed']

Prefix: 'He went to the'
Suggestions: ['north', 'United', 'west', 'east', 'south']

Prefix: 'The city of'
Suggestions: ['Omaha', 'Galveston', 'Hope', 'Newark', 'Zagreb']

Prefix: 'She said.'
Suggestions: ['"', 'Loose', 'She', 'City', 'The']

Prefix: "The player's"
Suggestions: ['actions', 'score', 'character', 'team', 'journey']



In [16]:
validation_results = evaluate_model(ngram_model, encoded_lines["validation"], top_k=5)

test_results = evaluate_model(ngram_model, encoded_lines["test"], top_k=5)

print("FINAL n-gram model results:")
print("=" * 55)

print("Model order:", ngram_model.order)
print("Vocabulary size:", vocab_size)
print("Selected alpha:", ngram_model.alpha)
print("Selected lambdas:", ngram_model.lambdas)

print("\n{:<25} {:>12} {:>12}".format("Metric", "Validation", "Test"))
print("-" * 55)

for metric in validation_results:
    val = validation_results[metric]
    test = test_results[metric]

    if "accuracy" in metric or metric == "unk_rate":
        val_str = f"{val:.2%}"
        test_str = f"{test:.2%}"

    elif metric == "number_of_predictions":
        val_str = str(val)
        test_str = str(test)

    else:
        val_str = f"{val:.4f}"
        test_str = f"{test:.4f}"

    print(f"{metric:<25} {val_str:>12} {test_str:>12}")


FINAL n-gram model results:
Model order: 3
Vocabulary size: 46165
Selected alpha: 5e-05
Selected lambdas: {2: 0.4, 1: 0.4, 0: 0.2}

Metric                      Validation         Test
-------------------------------------------------------
perplexity                    348.3720     338.6003
top1_accuracy                   17.88%       17.86%
top5_accuracy                   35.82%       35.79%
unk_rate                         5.38%        5.96%
number_of_predictions           206188       232065


In [17]:
for split, sequences in encoded_lines.items():
    total = sum(len(seq) - 1 for seq in sequences)
    unknowns = sum(token == unk_id for seq in sequences for token in seq[1:])
    print(f"{split}: {unknowns / total:.2%} <unk>")

test: 5.96% <unk>
train: 1.98% <unk>
validation: 5.38% <unk>
